# Khmer Handwriting Recognition  

## Extract Quantum-Ready Features

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("../../src").resolve()))
from cnn_model import CnnFeatureExtractor  
from config import CLASSICAL_MODELS_DIR, IMG_HEIGHT, IMG_WIDTH  # noqa: E402

READY_DIR = Path("../../data/3_ready/cnn_alphabet")

QUBIT_SIZES = [4, 6, 8]

df = pd.read_parquet(READY_DIR / "cleaned_dataset.parquet")
print(f"rows: {len(df)}, classes: {df['text'].nunique()}")

rows: 2640, classes: 33


## CNN architecture comes from src/cnn_model.py 

## For each qubit count, load its trained CNN and extract quantum-ready features 

In [3]:
def build_tensors(sub_df, label_map):
    flat = np.stack(sub_df["cleaned"].to_numpy())  # saved flattened -- parquet can't hold 2D arrays
    images = torch.tensor(flat.reshape(-1, IMG_HEIGHT, IMG_WIDTH), dtype=torch.float32).unsqueeze(1)
    labels = torch.tensor(sub_df["text"].map(label_map).to_numpy(), dtype=torch.long)
    return images, labels


def extract_features_for(n_qubits):
    name = f"{n_qubits}q"
    model_dir = CLASSICAL_MODELS_DIR / name
    with open(model_dir / "label_map.json", encoding="utf-8") as f:
        label_map = json.load(f)
    classes = set(label_map.keys())

    model = CnnFeatureExtractor(n_qubits, len(label_map))
    model.load_state_dict(torch.load(model_dir / "cnn.pt", map_location="cpu"))
    model.eval()

    sub_df = df[df["text"].isin(classes)]
    splits = {}
    for split_name in ["train", "val", "test"]:
        images, labels = build_tensors(sub_df[sub_df["split"] == split_name], label_map)
        with torch.no_grad():
            features = model.features(images).numpy()
        splits[split_name] = (features, labels.numpy())
        print(f"[{name}] {split_name}: {features.shape}")

    np.savez(
        READY_DIR / f"input_alphabet_{n_qubits}_qubits.npz",
        train_features=splits["train"][0], train_labels=splits["train"][1],
        val_features=splits["val"][0], val_labels=splits["val"][1],
        test_features=splits["test"][0], test_labels=splits["test"][1],
    )
    print(f"[{name}] Wrote: {READY_DIR / f'input_alphabet_{n_qubits}_qubits.npz'}\n")


for n_qubits in QUBIT_SIZES:
    extract_features_for(n_qubits)

[4q] train: (896, 4)
[4q] val: (192, 4)
[4q] test: (192, 4)
[4q] Wrote: ..\..\data\3_ready\cnn_alphabet\input_alphabet_4_qubits.npz

[6q] train: (1848, 6)
[6q] val: (396, 6)
[6q] test: (396, 6)
[6q] Wrote: ..\..\data\3_ready\cnn_alphabet\input_alphabet_6_qubits.npz

[8q] train: (1848, 8)
[8q] val: (396, 8)
[8q] test: (396, 8)
[8q] Wrote: ..\..\data\3_ready\cnn_alphabet\input_alphabet_8_qubits.npz

